# temas

In [ ]:

df_temas_unicos = df_temas[['codTema', 'tema']].drop_duplicates(subset='codTema')

for _, row in df_temas_unicos.iterrows():
    cursor.execute("""
        INSERT IGNORE INTO tema_proposicoes (cd_tema, nome)
        VALUES (%s, %s)
    """, (int(row['codTema']), row['tema']))

conn.commit()
print(f"Temas inseridos: {len(df_temas_unicos)}")

Temas inseridos: 32


TOP TEMAS

In [ ]:
# Contar quantas proposições por tema
df_contagem = df_temas.groupby(['codTema', 'tema']).size().reset_index(name='quantidade')

for _, row in df_contagem.iterrows():
    cursor.execute("""
        INSERT IGNORE INTO top_temas (cd_tp_temas, tipo, quantidade_pautas)
        VALUES (%s, %s, %s)
    """, (int(row['codTema']), row['tema'], int(row['quantidade'])))

conn.commit()
print(f"Top temas inseridos: {len(df_contagem)}")

Top temas inseridos: 32


  Temas deputados

Criando a tabela DEPUTADO_TEMA

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS deputado_tema (
    fk_deputado INT,
    fk_tema INT,
    qtd_proposicoes INT,
    PRIMARY KEY (fk_deputado, fk_tema)
)
""")

 CÉLULA 1 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
PASTA = '/content/drive/MyDrive/arquivos csv'


Ler todos CSVS

In [ ]:
import pandas as pd
import glob

def ler_csvs(pasta, padrao):
    """Lê todos os CSVs que casam com o padrão e concatena."""
    arquivos = glob.glob(os.path.join(pasta, padrao))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para: {padrao} em {pasta}")
    print(f"  [{padrao}] {len(arquivos)} arquivo(s): {[os.path.basename(a) for a in arquivos]}")
    return pd.concat([pd.read_csv(a, sep=';') for a in arquivos], ignore_index=True)

print(" Lendo CSVs...")
autores = ler_csvs(PASTA, 'proposicoesAutores*.csv')
temas   = ler_csvs(PASTA, 'proposicoesTemas*.csv')

print(f"\n autores: {len(autores):,} linhas | temas: {len(temas):,} linhas")


 Lendo CSVs...
  [proposicoesAutores*.csv] 4 arquivo(s): ['proposicoesAutores-2026.csv', 'proposicoesAutores-2025.csv', 'proposicoesAutores-2024.csv', 'proposicoesAutores-2023.csv']
  [proposicoesTemas*.csv] 5 arquivo(s): ['proposicoesTemas-2026.csv', 'proposicoesTemas-2025.csv', 'proposicoesTemas-2024.csv', 'proposicoesTemas-2023.csv', 'proposicoesTemas-2022.csv']

 autores: 440,214 linhas | temas: 107,030 linhas


Processamento deputados

In [ ]:
autores = autores[autores['codTipoAutor'] == 10000].copy()
print(f" Deputados filtrados: {len(autores):,} linhas")

# Extrair idProposicao do URI no arquivo de temas
temas['idProposicao'] = temas['uriProposicao'].str.split('/').str[-1].astype(int)

# JOIN autores ↔ temas via idProposicao
df = autores[['idProposicao', 'idDeputadoAutor']].merge(
    temas[['idProposicao', 'codTema']],
    on='idProposicao',
    how='inner'
)
print(f"🔗 Após JOIN: {len(df):,} combinações")

# Contar proposições por (deputado, tema) — TODOS os temas
resultado = (
    df.groupby(['idDeputadoAutor', 'codTema'])
    .size()
    .reset_index(name='qtd_proposicoes')
    .rename(columns={
        'idDeputadoAutor': 'fk_deputado',
        'codTema':         'fk_tema'
    })
    .sort_values(['fk_deputado', 'qtd_proposicoes'], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"\n {len(resultado):,} linhas no resultado (deputado + tema)")
print(resultado.head(10).to_string(index=False))

 Deputados filtrados: 401,990 linhas
🔗 Após JOIN: 152,002 combinações

 14,938 linhas no resultado (deputado + tema)
 fk_deputado  fk_tema  qtd_proposicoes
     62881.0       34               17
     62881.0       70               16
     62881.0       53               15
     62881.0       57               13
     62881.0       44               10
     62881.0       58               10
     62881.0       68                8
     62881.0       76                7
     62881.0       56                6
     62881.0       40                4


Adicionar ao banco

In [ ]:
valores = [
    (int(row.fk_deputado), int(row.fk_tema), int(row.qtd_proposicoes))
    for _, row in resultado.iterrows()
]

cursor.executemany(
    "INSERT INTO deputado_tema (fk_deputado, fk_tema, qtd_proposicoes) VALUES (%s, %s, %s)",
    valores
)
conn.commit()
print(f" {cursor.rowcount} linhas inseridas na tabela deputado_tema!")

 14938 linhas inseridas na tabela deputado_tema!
